In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 16
DEVICE = torch.device('cpu')

TEST_DIR = r'D:\ScanO-assign-Apaar\dataset\test'
OUTPUT_DIR = 'submission/outputs'

In [3]:
def build_df(root):

    paths = []
    labels = []

    for label, cls in enumerate(['normal', 'pneumonia']):

        cls_path = os.path.join(root, cls)

        for img in os.listdir(cls_path):
            paths.append(os.path.join(cls_path, img))
            labels.append(label)

    return pd.DataFrame({
        'image_path': paths,
        'label': labels
    })


test_df = build_df(TEST_DIR)

In [4]:
tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
class XrayDataset(Dataset):

    def __init__(self, df, tfms):
        self.df = df
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(row.image_path).convert('RGB')
        raw = img.resize((224, 224))

        x = self.tfms(img)

        return x, row.label, row.image_path, np.array(raw)

In [6]:
test_ds = XrayDataset(test_df, tfms)

loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [7]:
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(512, 2)

model.load_state_dict(
    torch.load(f'{OUTPUT_DIR}/best_model_phase2.pth', map_location='cpu')
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [8]:
activations = None
gradients = None


def forward_hook(module, inp, out):
    global activations
    activations = out


def backward_hook(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]


model.layer4.register_forward_hook(forward_hook)
model.layer4.register_full_backward_hook(backward_hook)

In [9]:
preds = []
trues = []
probs = []
records = []

for imgs, labels, paths, raws in tqdm(loader):

    imgs = imgs.to(DEVICE)

    outputs = model(imgs)

    p = F.softmax(outputs, dim=1)

    conf, pred = torch.max(p, 1)

    preds.extend(pred.numpy())
    trues.extend(labels.numpy())
    probs.extend(conf.detach().numpy())

    for i in range(len(paths)):

        records.append({
            'image_path': paths[i],
            'true_label': int(labels[i]),
            'predicted_label': int(pred[i]),
            'confidence': float(conf[i])
        })

  0%|          | 0/10 [00:00<?, ?it/s]C:\Users\apaar\AppData\Local\Temp\ipykernel_111372\399100315.py:26: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  'confidence': float(conf[i])
100%|██████████| 10/10 [00:11<00:00,  1.19s/it]


In [10]:
acc = accuracy_score(trues, preds)
prec = precision_score(trues, preds)
rec = recall_score(trues, preds)
f1 = f1_score(trues, preds)
cm = confusion_matrix(trues, preds)
roc = roc_auc_score(trues, preds)

with open(f'{OUTPUT_DIR}/metrics2.txt', 'a') as f:

    f.write('\nPHASE 1\n')
    f.write(f'Accuracy: {acc:.4f}\n')
    f.write(f'Precision: {prec:.4f}\n')
    f.write(f'Recall: {rec:.4f}\n')
    f.write(f'F1: {f1:.4f}\n')
    f.write(f'ROC-AUC: {roc:.4f}\n')
    f.write(f'Confusion Matrix:\n{cm}\n')

In [11]:
pd.DataFrame(records).to_csv(
    f'{OUTPUT_DIR}/predictions2.csv',
    index=False
)

In [12]:
def generate_gradcam(img_tensor, raw_img, pred_class):

    model.zero_grad()

    out = model(img_tensor)
    out[:, pred_class].backward()

    pooled = gradients.mean(dim=[0, 2, 3])

    acts = activations[0]

    for i in range(512):
        acts[i] *= pooled[i]

    heatmap = acts.mean(dim=0).detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= heatmap.max()

    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed = heatmap * 0.4 + raw_img

    return np.uint8(superimposed)

In [13]:
correct = []
wrong = []

for r in records:

    if r['true_label'] == r['predicted_label']:
        correct.append(r)
    else:
        wrong.append(r)

correct = sorted(correct, key=lambda x: -x['confidence'])[:3]
wrong = sorted(wrong, key=lambda x: x['confidence'])[:3]

samples = correct + wrong


for idx, row in enumerate(samples):

    img = Image.open(row['image_path']).convert('RGB')
    raw = np.array(img.resize((224, 224)))

    x = tfms(img).unsqueeze(0)

    pred = row['predicted_label']

    cam = generate_gradcam(x, raw, pred)

    # Generate standalone heatmap
    model.zero_grad()

    out = model(x)
    out[:, pred].backward()

    pooled = gradients.mean(dim=[0, 2, 3])
    acts = activations[0]

    for i in range(512):
        acts[i] *= pooled[i]

    heatmap = acts.mean(dim=0).detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= heatmap.max() + 1e-8

    heatmap_resized = cv2.resize(heatmap, (224, 224))

    # Plot format
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))

    ax[0].imshow(raw)
    ax[0].set_title('Original', fontsize=18)
    ax[0].axis('off')

    im = ax[1].imshow(heatmap_resized, cmap='jet')
    ax[1].set_title('Grad-CAM', fontsize=18)
    ax[1].axis('off')

    fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)

    ax[2].imshow(raw)
    ax[2].imshow(heatmap_resized, cmap='jet', alpha=0.45)
    ax[2].set_title('Overlay', fontsize=18)
    ax[2].axis('off')

    plt.tight_layout()

    save_path = f'{OUTPUT_DIR}/sample_outputs/phase2_gradcam_{idx}.png'

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()